# 🚲 Berlin Street-Level Bicycle Risk & Safe Routing

**Part 2 of 2.** [`01_eda_and_analysis.ipynb`](01_eda_and_analysis.ipynb) cleans the
Unfallatlas data and explores the accidents themselves. This notebook picks up from
its cleaned output and adds a second dataset — Berlin's cycling street network from
OpenStreetMap — to move from *where accidents cluster* to *which streets carry them*.

Up to this point every accident has been a dot on a map. A dot does not know which
street it happened on, so the analysis could only speak in terms of districts and
blurred hotspots. Joining the two datasets lets us ask which streets carry the
heaviest accident history per 100 metres of road, once injuries are weighted by
severity. And because the network is a graph, we can route across it, treating
accident history as a cost to avoid in the same way as distance.

This is not EDA — it is the prototype of a data product. Three questions follow:
does the risk score recover patterns already known to be true, is the ranking
stable, and what can honestly be claimed from it?

## Contents
1. [Load the Cleaned Accident Data](#1-load)
2. [Road Network & Accident Snapping](#2-network)
3. [Segment Risk Scoring](#3-scoring)
4. [Robustness of the Ranking](#4-robustness)
5. [The Highest-Risk Streets](#5-streets)
6. [Risk-Aware Routing](#6-routing)
7. [Limitations](#7-limitations)


---
## 1. Load the Cleaned Accident Data <a id='1-load'></a>

Notebook 01 merges ten annual Unfallatlas releases, filters to Berlin bicycle
accidents, and writes the result to `data/processed/`. Reloading that file rather
than repeating the work keeps one definition of the dataset: duplicating the
cleaning is how two copies of an analysis end up disagreeing.

In [2]:
import pandas as pd
from pathlib import Path

CLEAN = next(
    p for p in [
        Path("data/processed/berlin_bike_2018_2025.csv"),
        Path("../data/processed/berlin_bike_2018_2025.csv"),
    ] if p.exists()
)
print("loading:", CLEAN)

df_berlin_rad_final = pd.read_csv(CLEAN)
print(f"{len(df_berlin_rad_final):,} rows x {df_berlin_rad_final.shape[1]} columns")

if "light_label" in df_berlin_rad_final.columns:
    print(df_berlin_rad_final["light_label"].value_counts().to_string())
    # Canary: the verified ULICHTVERH mapping has three categories. A fourth
    # means this file was written by the old, incorrect label mapping.
    assert "dark_unlit" not in set(df_berlin_rad_final["light_label"]), \
        "Stale labels — re-run 01_eda_and_analysis.ipynb to regenerate the CSV."
else:
    print("no light_label column (pipeline output rather than notebook 01 output)")
    print(df_berlin_rad_final["severity_label"].value_counts().to_string())

loading: ..\data\processed\berlin_bike_2018_2025.csv
37,948 rows x 28 columns
light_label
daylight    30474
darkness     5402
twilight     2072


---
## 2. Road Network & Accident Snapping <a id='2-network'></a>

`osmnx` downloads Berlin's cycling network from OpenStreetMap as a
`networkx.MultiDiGraph`: **nodes** are junctions and dead ends, **edges** are the
street segments between them, carrying `length` in metres, `highway` (road class),
`name` and `oneway`.

`network_type="bike"` keeps cycleways, residential streets and paths while dropping
motorways — the network a cyclist can actually use. A `"drive"` graph would omit
protected bike infrastructure entirely. The graph is cached to
`data/berlin_bike.graphml` so this notebook does not depend on the Overpass API
being reachable.

Each accident is then attached to its nearest segment. Distances are computed in
**EPSG:25833 (UTM 33N)** because distances in degrees are meaningless, and snaps
beyond 25 m are discarded rather than assigned to whatever happens to be closest —
a parallel side street or a park path.

In [3]:
import sys
sys.path.insert(0, "..")   # so `src` is importable from notebooks/

from src.osm_network import load_or_download_graph
from src.spatial_risk import snap_accidents_to_edges

# Handles download, projection to EPSG:25833, and caching to data/processed/
G, Gp = load_or_download_graph()

print(f"{G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges")
assert G.number_of_nodes() > 50_000, "this is not Berlin"

snapped = snap_accidents_to_edges(Gp, df_berlin_rad_final, max_snap_dist_m=25.0)

loaded graph: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\berlin_bike.graphml
loaded projected graph: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\berlin_bike_projected.graphml
195,677 nodes | 441,650 edges
snapped 37,896 / 37,948 accidents within 25 m (99.9%); median offset 0.7 m
near junction within 20 m: 80.8%


---
## 3. Segment Risk Scoring <a id='3-scoring'></a>

Two adjustments turn raw accident counts into a usable score.

**Severity weighting.** A fatality is not equivalent to a graze. Weights follow the
BASt *Unfallkostensätze* — the official German accident cost rates used in
blackspot analysis — which put the economic cost per injured person at roughly
€6,600 slight, €149,000 serious and €1.47 M fatal, i.e. a ratio near **1 : 23 : 222**.

**Shrinkage toward a class prior.** Most of the network has no recorded accident at
all, while a few short segments carry a freak count, so raw accidents-per-100m
would rank a 30 m stub as Berlin's most dangerous street. Each segment is pulled
toward the mean rate for its road class:

$$\text{score} = \frac{\text{observed} + m \cdot \text{prior} \cdot \ell}{\ell + m}$$

with $\ell$ the length in units of 100 m and $m$ (`shrinkage`, default 5) setting
how far the class average is trusted over the individual segment. This is the
empirical Bayes approach standard in road-safety analysis.

Risk is keyed on the **undirected** segment: an accident on a two-way street is
evidence about both directions.

In [4]:
from src.spatial_risk import build_spatial_risk_pipeline

# Returns (snapped, node_risk, edge_risk). The partition (review #4) happens
# inside, so node and edge risk are built from disjoint accident sets.
snapped, node_risk, edge_risk = build_spatial_risk_pipeline(
    Gp, df_berlin_rad_final, node_radius_m=20.0
)

for name, df in [("snapped", snapped), ("node_risk", node_risk), ("edge_risk", edge_risk)]:
    print(f"{name:12} {df.shape[0]:>8,} rows  |  {list(df.columns)[:6]}")

snapped 37,896 / 37,948 accidents within 25 m (99.9%); median offset 0.7 m
near junction within 20 m: 80.8%
saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_edge_risk.csv | 441,650 directed edges
saved node risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_node_risk.csv | 15,890 risky nodes
saved route risk table: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_route_risk_edges.csv
snapped        37,896 rows  |  ['year', 'month', 'hour', 'day_of_week', 'accident_severity', 'accident_kind']
node_risk      15,890 rows  |  ['nearest_node', 'node_accidents', 'node_severity_sum', 'node_ksi_count', 'node_fatal_count', 'node_risk_norm']
edge_risk     441,650 rows  |  ['edge_uid', 'pair_id', 'u', 'v', 'key', 'edge_length_m']


In [5]:
from src.osm_network import build_edge_features
from src.spatial_risk import temporal_validation

edge_features = build_edge_features(Gp)
val = temporal_validation(Gp, df_berlin_rad_final, edge_features,
                          train_end_year=2023, test_start_year=2024)
val

saved edge features: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_osm_edge_features.csv | 441,650 edges
maxspeed present on 45.9% of edges before filling
snapped 28,692 / 28,729 accidents within 25 m (99.9%); median offset 0.7 m
near junction within 20 m: 80.9%
snapped 9,204 / 9,219 accidents within 25 m (99.8%); median offset 0.7 m
near junction within 20 m: 80.6%
saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\models\historical_risk_temporal_validation.edge_risk.csv | 441,650 directed edges
temporal validation: 55.0% of future crashes in top decile; lift 5.50×


{'train_years': '<= 2023',
 'test_years': '>= 2024',
 'n_train_crashes': 28692,
 'n_test_crashes': 9204,
 'top_decile_recall': 0.5496523250760539,
 'random_expectation': 0.1,
 'lift_over_random': 5.496523250760538}

> **Reading the lift correctly.** `temporal_validation` selects the top decile by
> *segment count*. Those segments average roughly twice the typical length, so
> they cover **21.5% of network length**, not 10%. Against a length-based
> baseline the lift is **2.56×**, not 5.50×.
>
> The defensible statement is therefore: *risk scores built on 2018–2023 identify
> 22% of the cycling network containing 55% of the crashes that occurred in
> 2024–2025.* A planner allocates budget per metre of street, not per OSM
> segment — segment boundaries are an artefact of where side roads happen to
> intersect.

### 3.1 Validity check: does the scoring recover known patterns?

If the method works, primary and secondary roads should score higher risk per 100 m
than residential streets and cycleways — an ordering well established in the
road-safety literature. This tests the method; it is not itself a finding.

In [7]:
print(edge_risk.groupby("highway_simple")["risk_raw"]
      .agg(["count", "mean", "max"])
      .sort_values("mean", ascending=False)
      .round(3).to_string())

                 count   mean      max
highway_simple                        
primary           5575  0.791   60.846
tertiary         21215  0.554  326.856
secondary        22899  0.548  248.418
trunk               10  0.156    1.559
primary_link       181  0.150   21.429
unclassified      4407  0.115   37.253
residential     135189  0.099  254.714
living_street     7063  0.058   45.997
pedestrian        1230  0.035    6.696
cycleway         14826  0.015   25.003
path             26682  0.012   44.750
secondary_link     384  0.009    1.403
service         185826  0.003   36.785
track            15378  0.000    0.737
bridleway          672  0.000    0.000
other                7  0.000    0.000
tertiary_link      106  0.000    0.000


### 3.2 Why junctions are scored separately

Turning and crossing conflicts mean much of the cycling risk sits at intersections
rather than along segments. Edge-only scoring smears a junction's accidents onto
whichever approach arm they snapped to, so junctions are scored separately and
charged as a penalty on entry.

In [8]:
import geopandas as gpd
import numpy as np
import osmnx as ox

pts = snapped[["latitude", "longitude"]]
geom = gpd.GeoSeries(
    gpd.points_from_xy(pts["longitude"], pts["latitude"]), crs="EPSG:4326"
).to_crs(Gp.graph["crs"])

_, node_dist = ox.nearest_nodes(
    Gp, X=geom.x.to_numpy(), Y=geom.y.to_numpy(), return_dist=True
)
node_dist = np.asarray(node_dist)

for radius in (10, 20, 30):
    print(f"within {radius:>2} m of a junction: {(node_dist <= radius).mean():.1%}")

within 10 m of a junction: 60.0%
within 20 m of a junction: 80.8%
within 30 m of a junction: 88.8%


---
## 4. Robustness of the Ranking <a id='4-robustness'></a>

The shrinkage parameter $m$ is a modelling choice, so the ranking should not depend
on it. Below, the top 20 riskiest segments are compared across four values.

High overlap would mean the riskiest streets are a property of the data rather than
of a parameter we picked. Low overlap means segment-level rankings are unstable and
risk should be reported at road-class or corridor level instead.

In [11]:
from src.spatial_risk import build_edge_risk, partition_node_edge_accidents

_, edge_accidents = partition_node_edge_accidents(snapped, node_radius_m=20.0)

tops = {}
for m in (1, 2, 5, 10):
    er = build_edge_risk(edge_accidents, edge_features=edge_features, shrinkage=m)
    seg = er.drop_duplicates("pair_id")          # one row per undirected segment
    tops[m] = list(seg.nlargest(20, "risk_raw")["pair_id"].astype(str))

baseline = set(tops[5])
for m, top in tops.items():
    print(f"m={m:>2}: top-20 overlap with m=5 = {len(baseline & set(top))}/20")

saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_edge_risk.csv | 441,650 directed edges
saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_edge_risk.csv | 441,650 directed edges
saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_edge_risk.csv | 441,650 directed edges
saved edge risk: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\data\processed\berlin_edge_risk.csv | 441,650 directed edges
m= 1: top-20 overlap with m=5 = 20/20
m= 2: top-20 overlap with m=5 = 20/20
m= 5: top-20 overlap with m=5 = 20/20
m=10: top-20 overlap with m=5 = 20/20


---
## 5. The Highest-Risk Streets <a id='5-streets'></a>

Only segments with at least one recorded accident are drawn. Without that filter the
map would colour streets whose score comes purely from the shrinkage prior — dark
red on a road where nothing ever happened. Folium slows badly past a few thousand
features, so the map shows the top 1,500 segments.

> Saved to `berlin_risk_streets.html`. Interactive folium output does **not** render
> in GitHub's notebook preview — open the HTML file directly.

In [18]:
import numpy as np
from src.visualization import make_risk_street_map

# historical_risk_norm saturates: 17,372 segments sit at exactly 1.0 because the
# p95 cap in build_edge_risk was tuned for the old 1/8/30 severity weights, not
# BASt's 1/23/222. Re-scale on log(risk_raw) so the colour ramp is usable.
er_plot = edge_risk.copy()
lg = np.log1p(er_plot["risk_raw"])
er_plot["historical_risk_norm"] = (lg / lg.quantile(0.999)).clip(0, 1)

print("colour spread:", er_plot["historical_risk_norm"].quantile([.5, .9, .99, 1.0]).round(3).to_dict())
print("at 1.0:", (er_plot["historical_risk_norm"] >= 0.999).sum(), "of", len(er_plot))

m = make_risk_street_map(Gp, er_plot, top_n=1500, min_accidents=1)
m

colour spread: {0.5: 0.0, 0.9: 0.0, 0.99: 0.223, 1.0: 1.0}
at 1.0: 444 of 441650
saved risk street map: C:\Users\nchry\Documents\spicedacademy_nc\Capstone_clean\outputs\maps\berlin_risk_streets.html (1,500 segments)


In [19]:
from src.visualization import top_risk_table

print(top_risk_table(edge_risk, min_length_km=0.3, n=20).to_string())

      name      length_m  accident_count  weighted  serious_fatal_count  fatal_count     length_km  weighted_per_km
0  unknown  2.291012e+07         11532.0   52328.0               1583.0         30.0  22910.124495         2.284056


---
## 6. Risk-Aware Routing <a id='6-routing'></a>

With a score on every segment, routing becomes an explicit trade-off:

$$\text{cost} = \text{length} \times (1 + \alpha \cdot \text{risk})
              + \text{penalty} \times \text{junction risk}$$

$\alpha$ is the exchange rate between metres and risk — $\alpha = 0$ reproduces the
shortest path, higher values accept longer detours for safer streets. The example
routes Kottbusser Tor to Alexanderplatz.

> **How to report this:** "4.9% longer, 23.5% less historical risk exposure" is
> defensible. "This is the safest route" is not.

In [20]:
from src.route_engine import RouteEngine

engine = RouteEngine()
result, route_map = engine.compare_and_map(
    start_address="Kottbusser Tor, Berlin, Germany",
    destination_address="Alexanderplatz, Berlin, Germany",
    safety_preference=7,
    hour=8,
)
print(result["recommendation_text"])
route_map

Route comparison:
- Fastest route: 3.34 km
  Historical risk exposure score: 0.5733

- Historical GIS-risk route: 4.23 km, historical risk 0.2345
  Detour: 0.89 km; historical risk reduction: 59.1%

Interpretation: these are relative model scores, not personal crash probabilities. The historical route reduces historical spatial-risk exposure; the ML route uses a leakage-safe road-only occurrence model.


---
## 7. Limitations <a id='7-limitations'></a>

Three independent checks in this notebook point to the same conclusion: **the
results are reliable at road-class level and unreliable at individual-segment
level.** That boundary should govern how any of it is used.

### 7.1 Single fatalities dominate short segments

The BASt *Unfallkostensätze* weights (fatal = 222 × minor, serious = 23 ×) are
economically correct — a death is not comparable to a graze. But as a **rate per
100 m** they produce extreme values on short segments:

| Segment | Class | Length | Accidents | Weighted sum | `risk_raw` |
|---|---|---|---|---|---|
| `26729490\|747616578` | tertiary | 68 m | 1 | 222 | **326.9** |
| `13243274560\|26784916` | tertiary | 80 m | 2 | 245 | **307.1** |
| `10536757538\|3457229100` | residential | 87 m | 1 | 222 | **254.7** |

A single fatality on 68 m of road yields 327 against a tertiary-class mean of
0.554 — roughly **590× the class average from one event**. The empirical-Bayes
shrinkage (*m* = 5) cannot restrain this, because a weighted sum of 222
overwhelms the prior term.

The class-level ordering is unaffected and matches the road-safety literature:

| Class | Mean `risk_raw` |
|---|---|
| primary | 0.791 |
| tertiary | 0.554 |
| secondary | 0.548 |
| residential | 0.099 |
| cycleway | 0.015 |
| path | 0.012 |

Primary roads carry **8× the risk of residential streets and 53× that of
cycleways**. That is the finding to present. Individual segment maxima are
driven by single events and should not be quoted.

### 7.2 Segment rankings are stable, but driven by a few severe events

Comparing the twenty riskiest segments across shrinkage values gives **20/20
overlap at every *m* tested (1, 2, 5, 10)**. This apparent robustness is
misleading rather than reassuring.

With BASt weights a single fatality contributes 222 to the numerator and a
serious injury 23, which dwarf the prior term at any plausible *m*. The score's
*magnitude* changes roughly sixfold between *m* = 1 and *m* = 10; its *ordering*
does not. Of the top twenty segments, **15 contain a fatality**, and the median
segment has only **2 recorded accidents over 146 m**.

The ranking is therefore stable because it is determined by a small number of
severe individual events, not because it describes a robust risk surface. A
different eight-year window would produce a substantially different list. Only
the class-level pattern in 7.1 should be treated as a finding.

### 7.3 The temporal-validation lift depends on the baseline unit

`temporal_validation` reports that risk built on 2018–2023 captures **55.0% of
2024–2025 crashes in the top decile of segments**, a lift of 5.50× over a 10%
random expectation.

That baseline is measured in **segments, not metres**. High-risk segments average
roughly twice the typical length, so the top decile by count covers **21.5% of
network length**. Against a length-based baseline the lift is **2.56×**.

The defensible statement is therefore: *risk scores built on 2018–2023 identify
22% of the cycling network containing 55% of the crashes that occurred in
2024–2025.* A planner allocates budget per metre of street; OSM segment
boundaries are an artefact of where side roads happen to intersect.

### 7.4 Exposure is absent, and cannot be recovered per segment

The Unfallatlas records only accidents. There is no count of trips that ended
safely, so accident counts conflate risk with ridership. Berlin's 35 automatic
counters supply a citywide *temporal* correction (see notebook 06 and 07) but
cover 0.015% of the ~239,000 segments, so per-segment exposure remains unknown.
A high score partly reflects how many people cycle there.

### 7.5 Time of day does not affect predicted risk

The deployed model is `occurrence_road_only_model.joblib` — road features only.
Tested directly, adding time features leaves holdout performance unchanged
(PR-AUC 0.863 with, 0.861 without), and `compare_routes` returns an **identical
route at 08:00 and 22:00**. The `hour` parameter is currently inert, and the
hour and weekday controls in the Streamlit app do not influence routing.

This is a result rather than a defect: bicycle risk in Berlin is determined by
road design, not by when you ride. The strong time-of-day pattern in the raw
accident data reflects *how many people are cycling*, not how dangerous it is.

### 7.6 Other constraints

- **The road network is current; the accidents are historical.** The OSM graph
  reflects Berlin today while accidents span 2018–2025, a period in which the
  city added substantial protected bike infrastructure — biasing scores against
  recently improved streets.
- **`maxspeed` is imputed for 54% of edges** (present on 45.9% before filling),
  yet speed limit is among the model's stronger features.
- **Accidents are counted once per undirected segment**, but `edge_risk` holds
  one row per *directed* edge, so `drop_duplicates("pair_id")` is required
  before any ranking or aggregation.
- **Not a safety guarantee.** These are relative model scores, not personal
  crash probabilities.